# IMPORT

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import folium
import csv
import json
import os

# DEFINE FOLDERS

In [ ]:
JSON_INPUT_FOLDER = Path("correlated_data_json")
DATASET_INPUT_FOLDER = Path("correlated_data_csv")
DATASET_OUTPUT_FOLDER = Path("clean_data_csv")
DELETED_STATIONS_FOLDER = Path("deleted_stations")
MAP_FOLDER = Path("maps")

folders = [JSON_INPUT_FOLDER, DATASET_INPUT_FOLDER, DATASET_OUTPUT_FOLDER, DELETED_STATIONS_FOLDER, MAP_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# DEFINE CONSTANTS

In [ ]:
REMAKE_CSV = False # To get the dataset from json files(will be saved in a csv file)
CSV_FILE_NAME = "correlated_data_2" #To save the dataset from json to csv/to get the dataset from csv
FINAL_CSV_FILE_NAME = "clean_data" # The prepared dataset


# DEFINE FUNCS

In [ ]:
def print_df(dataframe):
    print("=========================================")
    print(f"Shape: \nrows:     {dataframe.shape[0]}\nfeatures: {dataframe.shape[1]}")
    
    data_size_bytes = dataframe.memory_usage(deep=True).sum()
    data_size_mb = data_size_bytes / (1024**2)
    print(f"Dataframe size: {data_size_bytes} bytes ({data_size_mb:.2f} MB)")

    print("-----------------------")
    display(dataframe.dtypes)
    print("-----------------------")
    display(dataframe.describe())
    print("=========================================")

MISSING_STRINGS = {"", "na", "n/a", "null", "none", "nan", "-"}
def rows_with_missing(df):
    mask = df.isna().any(axis=1)

    for col in df.columns:
        if df[col].dtype == object:
            col_str = df[col].str.strip().str.lower()
            mask |= col_str.isin(MISSING_STRINGS)

    return df[mask]


def print_missing(df, max_rows=10):
    print("=========================================")
    display(df.isna().sum().sort_values(ascending=False))
    print("-----------------------")

    missing = rows_with_missing(df)

    display(missing.head(max_rows))
    print(f"Missing rows (total): {len(missing)}")
    print("=========================================")



# IF WANT TO REMAKE CSV FILE FROM JSON FILES

In [ ]:
CHUNK_SIZE = 50_000   # chunk size to save each time to CSV_FILE_NAME, so the memory doesn't explode
first_write = True

csv_path = os.path.join(DATASET_INPUT_FOLDER, f"{CSV_FILE_NAME}.csv")
total_file_num = len(os.listdir(JSON_INPUT_FOLDER))

if REMAKE_CSV:
    for file_idx, filename in enumerate(os.listdir(JSON_INPUT_FOLDER), start=1):
        if not filename.endswith(".json"):
            continue

        path = os.path.join(JSON_INPUT_FOLDER, filename)

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        rows = []

        for entry in data:
            row = {}

            # Station fields
            fields = entry.get("fields", {})
            for key, value in fields.items():
                if key == "coordonnees_geo" and isinstance(value, list) and len(value) == 2:
                    row["latitude"] = value[0]
                    row["longitude"] = value[1]
                else:
                    row[key] = value

            # Weather fields
            weather = entry.get("weather", {})
            if "time" in weather:
                row["weather_time"] = weather["time"]

            for key, value in weather.items():
                if key != "time":
                    row[f"weather_{key}"] = value

            row["source_file"] = filename
            rows.append(row)

            # Write chunk
            if len(rows) >= CHUNK_SIZE:
                df_chunk = pd.DataFrame(rows)
                df_chunk.to_csv(
                    csv_path,
                    mode="w" if first_write else "a",
                    header=first_write,
                    index=False,
                    encoding="utf-8"
                )
                first_write = False
                rows.clear()

        # Flush remaining rows of this file
        if rows:
            df_chunk = pd.DataFrame(rows)
            df_chunk.to_csv(
                csv_path,
                mode="w" if first_write else "a",
                header=first_write,
                index=False,
                encoding="utf-8"
            )
            first_write = False

        if file_idx % 100 == 0:
            print(f"{file_idx}/{total_file_num} JSON files processed")

    print("CSV generation finished.")

# LOAD DATA

In [ ]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/{CSV_FILE_NAME}.csv")
df.head(5)

In [ ]:
print_df(df)
print_missing(df)

# RENAME FEATURES(COLUMNS)

In [ ]:
feature_name_dict = {
    "name": "station_name",
    "stationcode": "station_code",
    "numbikesavailable": "total_bikes_now",
    "numdocksavailable": "total_bikes_taken",
    "nom_arrondissement_communes": "commune_name",
    "code_insee_commune": "postal_code",
    "weather_time": "timestamp",
    "weather_temperature_2m": "temperature",
    "weather_is_day": "is_day",
    "weather_wind_speed_10m": "wind_speed",
    "weather_wind_direction_10m": "wind_direction",
    "weather_weather_code": "weather_code"
}

df = df.rename(columns=feature_name_dict)

# CHECK

In [ ]:
print(df.isna().sum().sort_values(ascending=False))

# DROP INVALID ROWS(DOESNT CONTAIN THE DATA FOR TOTAL_BIKES_NOW)

In [ ]:
essential_data = [
    "station_name", 
    "station_code", 
    "postal_code", 
    "capacity", 
    "total_bikes_now", 
    "total_bikes_taken", 
    "timestamp"
]

df = df.dropna(subset=essential_data)

# RECHECK

In [ ]:
print_missing(df)

# KEEPING THE USABLE COLUMNS

In [ ]:
# Keep only useful columns
keep_cols = [
    "station_code",
    "latitude",
    "longitude",
    "capacity",
    "total_bikes_now",
    "total_bikes_taken",
    "timestamp",
    "temperature",
    "wind_speed",
    "wind_direction",
    "weather_code",
    "is_day",
]

df_cleaned = df[keep_cols].copy(deep=True)

print("Initial shape:", df_cleaned.shape)
print_df(df_cleaned)

# WEATHER TIME HANDLING

In [ ]:
df_cleaned["timestamp"] = pd.to_datetime(df_cleaned["timestamp"], errors="coerce")

# Drop rows without time
df_cleaned = df_cleaned.dropna(subset=["timestamp"])

# Time features
df_cleaned["hour"] = df_cleaned["timestamp"].dt.hour
df_cleaned["dayofweek"] = df_cleaned["timestamp"].dt.dayofweek
df_cleaned["is_weekend"] = (df_cleaned["dayofweek"] >= 5).astype(int)

# Cyclical encoding
df_cleaned["sin_hour"] = np.sin(2 * np.pi * df_cleaned["hour"] / 24)
df_cleaned["cos_hour"] = np.cos(2 * np.pi * df_cleaned["hour"] / 24)

In [ ]:
print_df(df_cleaned)

# CHANGE FEATURE DATA TYPES FOR MEMORY OPTIMISATION

In [ ]:
df_cleaned = df_cleaned.astype({
    "station_code": "uint32",
    "latitude": "float64",
    "longitude": "float64",
    "capacity": "uint8",
    "total_bikes_now": "uint8",
    "total_bikes_taken": "uint8",
    "temperature": "float16",
    "wind_speed": "float16",
    "wind_direction": "float16",
    "weather_code": "uint8",
    "is_day": "uint8",
    "hour": "uint8",
    "dayofweek": "uint8",
    "is_weekend": "uint8",
    "sin_hour": "float32",
    "cos_hour": "float32"
})

print_df(df_cleaned)
df_cleaned.head()

# HISTOGRAM OF ROWS

In [ ]:
df_counts = df_cleaned["station_code"].value_counts().reset_index()
df_counts.columns = ["station_code", "count"]
print(df_counts)

In [ ]:

rows = df_cleaned["station_code"].value_counts().values
print(rows)
drows = {v.item(): sum([1 for e in rows if e == v]) for v in rows}
bins, counts = list(drows.values()), list(drows.keys())
print(f"counts({len(counts)}): {counts}")
print(f"bins({len(bins)}): {bins}")


counts = df["station_code"].value_counts()   # index=station_code, value=#rows
count_distribution = counts.value_counts().sort_index()
print(count_distribution)
count_distribution.plot(kind="bar")
plt.xlabel("Rows per station_code")
plt.ylabel("Number of station_codes")
plt.title("Distribution of rows per station_code")
plt.tight_layout()
plt.show()

# MAKE THE MAP BEFORE DELETING STATIONS

In [ ]:
def save_unique_stations_map_html(
    df,
    out_html="idf_stations",
    station_col="station_code",
    lat_col="latitude",
    lon_col="longitude",
    radius=0.01,               
    color="red",
    fill_opacity=0.8,
    zoom_start=10,
    tiles="CartoDB positronNoLabels",
):
    # Keep only needed cols, drop missing coords
    d = df[[station_col, lat_col, lon_col]].dropna(subset=[lat_col, lon_col])

    # Keep one row per station (huge speedup vs millions of rows)
    d = d.drop_duplicates(subset=[station_col], keep="first")

    # Ensure numeric
    d[lat_col] = d[lat_col].astype(float)
    d[lon_col] = d[lon_col].astype(float)

    # Center map
    center_lat = float(d[lat_col].mean())
    center_lon = float(d[lon_col].mean())

    # Minimal basemap
    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start, tiles=tiles)

    # Plot points
    for code, la, lo in zip(d[station_col], d[lat_col], d[lon_col]):
        folium.CircleMarker(
            location=[la, lo],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=fill_opacity,
            tooltip=str(code),
        ).add_to(m)
    out_path = f"{MAP_FOLDER}/{out_html}.html"
    m.save(out_path)
    return out_path, len(d)


In [ ]:
html_path, n = save_unique_stations_map_html(df_cleaned, out_html="idf_stations_before", radius=0.01)
print("Saved:", html_path, "| stations mapped:", n)

# DETECT/REMOVE BROKEN STATIONS

In [ ]:
MIN_ROWS = 3999 # 3234
cols_to_check = ["total_bikes_now", "total_bikes_taken"]

# MAYBE WE SHOULD USE QUANTILE FOR THIS AS WELL, LIKE AT MOST N% SHOULD BE DELETED ETC

# Compute count, mean, and std per station
station_stats = df_cleaned.groupby("station_code")[cols_to_check].agg(["count", "std", "mean"])

# Identify bad stations: too few rows, NaN std, 0 std, or 0 mean
bad_stations = station_stats[
    (station_stats[("total_bikes_now", "count")] < MIN_ROWS) |
    (station_stats[("total_bikes_now", "std")].isna()) |
    (station_stats[("total_bikes_now", "std")] == 0) |
    (station_stats[("total_bikes_now", "mean")] == 0)
].index.tolist()

print(f"Number of stations to delete: {len(bad_stations)}")

# Stations to keep
all_stations = set(df_cleaned["station_code"].unique())
to_have = list(all_stations - set(bad_stations))
print(f"Number of stations to keep: {len(to_have)}")


# REMOVE THE BROKEN STATIONS

In [ ]:
print_df(df_cleaned)

df_removed = df_cleaned[df_cleaned.station_code.isin(to_have)]

print_df(df_removed)

# FILL MISSING WEATHER VALUES

In [ ]:
"""

weather_cols = [
    "temperature",
    "wind_speed",
    "wind_direction",
    "weather_code",
    "is_day",
]

for col in weather_cols:
    df[col] = df.groupby("station_code")[col].transform(
        lambda x: x.fillna(x.mean())
    )
"""

weather_float = ["temperature", "wind_speed", "wind_direction"]
weather_uint = ["weather_code", "is_day"]

# Fill float columns with station mean
for col in weather_float:
    df_removed[col] = df_removed.groupby("station_code")[col].transform(lambda x: x.fillna(x.mean()))

# Fill categorical columns with station mode
for col in weather_uint:
    df_removed[col] = df_removed.groupby("station_code")[col].transform(lambda x: x.fillna(x.mode()[0])).astype("uint8")


# MAKE THE MAP AFTER DELETING 

In [ ]:
html_path, n = save_unique_stations_map_html(df_removed, radius=0.01, out_html="idf_stations_removed")
print("Saved:", html_path, "| stations mapped:", n)

# SAVE THE BROKEN STATIONS IN A FILE

In [ ]:
broken_station_path = f"{DELETED_STATIONS_FOLDER}/broken_stations.csv"
stats = df_cleaned[df_cleaned["station_code"].isin(bad_stations)].groupby("station_code")["total_bikes_now"].agg(["mean", "std"])

with open(broken_station_path, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    # header
    writer.writerow(["station_code", "mean_total_bikes_now", "std_total_bikes_now"])
    for station, row in stats.iterrows():
        writer.writerow([station, row["mean"], row["std"]])

print(f"Broken stations saved: {len(bad_stations)}")
print(f"Saved: {broken_station_path}")

# CHECK

In [ ]:
print(df_removed.isna().sum().sort_values(ascending=False))

# DETAILS

In [ ]:
df_removed.describe()

# SAVE THE FINAL CLEAN DATASET

In [ ]:
df_cleaned.to_csv(f"{DATASET_OUTPUT_FOLDER}/{FINAL_CSV_FILE_NAME}.csv", index=False)
print(f"Clean dataset saved: {FINAL_CSV_FILE_NAME}.csv")